# 1. 理解工具

In [ ]:

from langchain.tools import tool
help(tool)

Help on function tool in module langchain_core.tools.convert:

tool(
    name_or_callable: str | collections.abc.Callable | None = None,
    runnable: langchain_core.runnables.base.Runnable | None = None,
    *args: Any,
    description: str | None = None,
    return_direct: bool = False,
    args_schema: type[pydantic.main.BaseModel] | dict[str, typing.Any] | None = None,
    infer_schema: bool = True,
    response_format: Literal['content', 'content_and_artifact'] = 'content',
    parse_docstring: bool = False,
    error_on_invalid_docstring: bool = True,
    extras: dict[str, typing.Any] | None = None
) -> langchain_core.tools.base.BaseTool | collections.abc.Callable[[collections.abc.Callable | langchain_core.runnables.base.Runnable], langchain_core.tools.base.BaseTool]
    Convert Python functions and `Runnables` to LangChain tools.

    Can be used as a decorator with or without arguments to create tools from functions.

    Functions can have any signature - the tool will automat

In [14]:
from langchain.tools import tool


@tool
def calcutor(op:str, a:float, b:float)-> float:
    """
    执行数学运算,包含四则运算
    """

    return 1   

print(type(calcutor))

print(calcutor.description) #calcutor.description

print(calcutor.func) #calcutor.func

print(calcutor.name) #calcutor.name

print(calcutor.args_schema) #calcutor.args_schema


<class 'langchain_core.tools.structured.StructuredTool'>
执行数学运算,包含四则运算
<function calcutor at 0x12ba6bf60>
calcutor
<class 'langchain_core.utils.pydantic.calcutor'>


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="ollama:qwen3.5:0.8b")
print(type(model))

model_with_tools = model.bind_tools([calcutor])
print(type(model_with_tools))

response = model_with_tools.invoke("计算 45与55的和")
print(response)
print(response.tool_calls)
print(type(response))

if response.tool_calls:
    print("调用工具",response.tool_calls[0]["args"])
else:
    print("直接结果",response.text)


<class 'langchain_ollama.chat_models.ChatOllama'>
<class 'langchain_core.runnables.base.RunnableBinding'>
content='' additional_kwargs={} response_metadata={'model': 'qwen3.5:0.8b', 'created_at': '2026-04-09T07:08:48.857813Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3793804334, 'load_duration': 141125834, 'prompt_eval_count': 302, 'prompt_eval_duration': 668419125, 'eval_count': 74, 'eval_duration': 2959457418, 'logprobs': None, 'model_name': 'qwen3.5:0.8b', 'model_provider': 'ollama'} id='lc_run--019d7112-687a-7213-b8ae-dce8c0643846-0' tool_calls=[{'name': 'calcutor', 'args': {'op': 'add', 'a': 45, 'b': 55}, 'id': 'c96be831-b7bd-47b1-9d86-f314f96f0e89', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 302, 'output_tokens': 74, 'total_tokens': 376}
[{'name': 'calcutor', 'args': {'op': 'add', 'a': 45, 'b': 55}, 'id': 'c96be831-b7bd-47b1-9d86-f314f96f0e89', 'type': 'tool_call'}]
<class 'langchain_core.messages.ai.AIMessage'>
调用工具 {'op': 'add', 'a

In [30]:
from langchain_core.tools.structured import StructuredTool
help(StructuredTool)

Help on class StructuredTool in module langchain_core.tools.structured:

class StructuredTool(langchain_core.tools.base.BaseTool)
 |  StructuredTool(
 |      *,
 |      name: str,
 |      description: str = '',
 |      args_schema: Annotated[type[pydantic.main.BaseModel] | dict[str, Any], SkipValidation()],
 |      return_direct: bool = False,
 |      verbose: bool = False,
 |      callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None,
 |      tags: list[str] | None = None,
 |      metadata: dict[str, typing.Any] | None = None,
 |      handle_tool_error: bool | str | collections.abc.Callable[[langchain_core.tools.base.ToolException], str] | None = False,
 |      handle_validation_error: bool | str | collections.abc.Callable[[pydantic_core._pydantic_core.ValidationError | pydantic.v1.error_wrappers.ValidationError], str] | None = False,
 |      response_format: Literal['content', 'content_and_artifact'] = 'co

In [ ]:
from langchain_core.tools.structured import StructuredTool
from pydantic import BaseModel, Field


def get_info():
    return "获取信息"

t1 = StructuredTool.from_function(
    func="",
    name="",
    description=""
    args_shema=""
)

@tool("查询天气"，description="查询天气")
def get_weather(data:str)->str:
    """
        执行天气的查询，参数是字符串格式的日期
    """
    return "hello"

print(get_weather.name)
print(get_weather.func)
print(get_weather.description)
print(get_weather.args_shema)


In [ ]:
from langchain_core.tools.structured import StructuredTool
from pydantic import BaseModel, Field


class InfoSchema(BaseModel):
    x:int = Field( description="订单号")
    y:int = Field( description="订单值")

def get_info():
    return "获取信息"

t1 = StructuredTool.from_function(  #
    func="",
    name="",
    description=""
    args_shema=""
)

@tool("查询天气"，description="查询天气")
def get_weather(data:str)->str:
    """
        执行天气的查询，参数是字符串格式的日期
    """
    return "hello"

print(t1.invoke())


In [ ]:
from langchain.tools import ToolRuntime
from langchain_core.runnables import RunnableConfig
@tool(description="获取信息")
def get_message(runtime:ToolRuntime,config:RunnableConfig)->str:        #runtime,config 是智能体传过来的 全局
    """
        获取用户的存储信息，以及最近用户访问的数据
    """
    print("*"*80)
    print(runtime.state)
    print("*"*80)
    print(runtime.context)
    print("*"*80)
    print(runtime.store)
    print("*"*80)
    print(runtime.config)
    print("*"*80)
    return "hello"    # 返回一个简单的问候语"hello"

In [ ]:
# 创建模型（省略）

# 创建智能体
from langchain.agents import create_agent

agent = create_agent(
    "ollama:qwen3.5:0.8b", 
    tools=[get_message], 
    system_prompt="你是数据处理专家"
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",  # 这里的引号应该是双引号，而不是单引号
            "content": "请给我查询用户存储的结果"
        }
    ]
})


# 2. 理解智能体